# JavaScript — DOM events

> **How this topic works**
> 1. **This notebook** — read the theory.
> 2. **`project/`** — a real Vite app where you do the exercise in the browser.
>
> Read this first, then follow the steps at the bottom. DOM snippets are shown as
> plain code blocks (this kernel has no browser); runnable cells are marked.

## LESSON 45 — DOM events

An **event** is something that happens: a click, a keystroke, a form submit. You react by registering a **listener**.

```js
element.addEventListener("click", () => {
  console.log("clicked");
});
```

### The event object

The callback receives an object describing what happened.

```js
button.addEventListener("click", (event) => {
  event.type;              // "click"
  event.target;            // the element it came from
  event.preventDefault();  // cancel the browser's default behaviour
});
```

On an input, the same object is how you read what was typed:

```js
input.addEventListener("input", (event) => {
  event.target.value;      // the current text in the field
});
```

Common event names: `click`, `input`, `submit`, `keydown`, `change`.

### Event delegation

Instead of one listener per child, put **one on the parent** and ask the event where it came from.

```js
list.addEventListener("click", (event) => {
  if (event.target.tagName !== "LI") return;
  console.log(event.target.textContent);
});
```

Why it matters: this keeps working for items **created later**. You cannot attach a listener to an element that does not exist yet, and delegation means you never need to. You'll need this in LESSON 46.

### closest — walking up from what was clicked

Delegation gives you `event.target`, which is the deepest element the click landed on. When every row is a plain `<li>` — as in this lesson's demo — that *is* the `<li>`, and the `tagName` check above is enough.

It stops being enough as soon as a row contains another element. If a list item wraps its text in a `<span>`, clicking that text gives you the span, and the check above rejects it.

`closest(selector)` starts at an element and walks **up** until it finds a match.

```js
list.addEventListener("click", (event) => {
  const item = event.target.closest("li");
  if (!item) return;                    // the click missed every li
  console.log(item.textContent);
});
```

Reach for `closest()` whenever a row might contain other elements.

### Keyboard events

```js
input.addEventListener("keydown", (event) => {
  if (event.key === "Enter") console.log("Enter pressed");
  if (event.key === "Escape") input.value = "";
});
```

`event.key` is the character or the key name — `"a"`, `"Enter"`, `"ArrowUp"`. `keydown` fires when the key goes down, `keyup` when it is released.

Do not use a keyboard listener to replace a form's `submit` event: in a form with a submit button, pressing Enter in a text field already submits it, and a `<button>` already reacts to Enter and Space. Add a keyboard handler for behaviour the browser does not already give you, not for what it does.

### Key notes

- **Pass the function, don't call it.** `addEventListener("click", handle)` — no parentheses. With `handle()` you run it immediately and register its *return value*.
- Two listeners on the same element both run. The second doesn't replace the first.
- `classList.toggle()` returns `true` if the class is now present — use that instead of tracking the state in your own variable.
- `preventDefault()` stops the browser's default action: a link navigating, a form reloading the page.
- **`closest()` includes the element itself.** If `event.target` is already the `<li>`, it returns that. It returns `null` when nothing up the tree matches, so check for `null`.
- **`tagName` comes back in UPPERCASE.** It is `"LI"`, never `"li"` — compare against the wrong case and the listener silently does nothing.
- **`target` vs `currentTarget`.** `target` is where the event started; `currentTarget` is the element the listener sits on. In delegation they differ, and that is the whole point.
- `event.key` is a **string**, and it is case-sensitive: `"Enter"`, not `"enter"`. For letters it reports the character actually produced, so holding Shift gives you `"A"` rather than `"a"`.

### Passing vs calling a function — runnable

This is the classic mistake of this lesson, and it has nothing to do with the browser —
so you can see it right here. Look at what each variable actually holds.

In [ ]:
function handleClick() {
  return "I am the result";
}

const passed = handleClick;    // no parentheses: the function itself
const called = handleClick();  // parentheses: it RUNS, now

console.log(typeof passed, "->", passed);
console.log(typeof called, "->", called);

// addEventListener needs something it can call LATER, when the click happens.
// Only one of these two is still callable:
console.log(passed());

---

## Now do the exercise

**1. Start the project**

```bash
cd project
npm install     # only the first time
npm run dev
```

Open the URL Vite prints, and open DevTools with **F12** — the demo logs there.

**2. Read the live demo**

Open `project/src/lessons/lesson-45-dom-events.js` and click around the page while
watching the console. Try the list (delegation) and the link (preventDefault).

**3. Do the exercise**

Open `project/src/exercise/exercise.js` and work through the numbered STEPs.
STEPs 1-4 build the theme toggle. STEPs 5-6 come back to delegation: one listener
on the list, and `closest()` to find the row that was clicked. STEP 7 clears the
highlights with the Escape key.

**Done when:** the button switches the page between light and dark and its label
always says what the *next* click will do, clicking any list item highlights it —
including the row whose text sits inside a `<span>` — and Escape clears them all.

Stuck? Paste `ai-prompt.txt` into a fresh AI session. The answer is in
`project/src/exercise/solution.js` — last resort.